In [13]:
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "absorption_paper_source_data").is_dir())
import sys
from pathlib import Path
import os
import importlib
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from abtem.core.energy import energy2wavelength, energy2sigma
from omegaconf import OmegaConf, DictConfig
from tqdm import tqdm

project_root = Path(str(REPO_ROOT / "diffBloch_version_0.0.1")).resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"[path] Added project root: {project_root}")

import diffBloch
diffbloch_path = getattr(diffBloch, "__file__", None)
if diffbloch_path is None:
    diffbloch_path = next(iter(getattr(diffBloch, "__path__", [])), "unknown")
print(f"[import] diffBloch loaded from: {Path(diffbloch_path).resolve()}")

from diffBloch.rotation_dataset import get_exp_ints, get_dataloaders
from diffBloch.dynamical import StructureFactorNet, BlochNet, ApparentThicknessNN
from diffBloch.atoms import Atoms
from diffBloch.utils import (
    initialize_scaling_factor,
    load_checkpoint,
    save_checkpoint,
    resolution_filter_diffraction_intensities,
    mask_structure_factors,
)
from diffBloch.optimize import init_optim


[import] diffBloch loaded from: diffBloch_version_0.0.1/diffBloch


In [47]:
cfg_atoms = OmegaConf.load("../configs/atoms/base.yaml")
ge_cif_path = project_root / "data/non_centro_vs_centro/Ge.cif"
ge_pets_path = project_root / "data/non_centro_vs_centro/Ge.cif_pets"


cfg_atoms["data"]["cif_file_path"] = str(ge_cif_path)
atoms_nn = Atoms(cfg_atoms)

cfg_refinement = OmegaConf.load("../configs/refinement/base.yaml")
cfg_refinement["dataloader"]["ignore_orientations"] = []
cfg_refinement["data"]["pets_path"] = str(ge_pets_path)
cfg_refinement["data"]["integration_semiangle"] = 1.5
cfg_refinement["data"]["rocking_curve_sampling"] = 60
cfg_refinement['data']['dsg'] = 0.00
cfg_refinement['data']['rsg'] = 0.9
semiangle = cfg_refinement["data"]["integration_semiangle"]

train_dataloader, val_dataloader = get_dataloaders(cfg_refinement, default_thickness=[500])

cfg_sf = OmegaConf.load("../configs/structure_factor/base.yaml")
cfg_thickness = OmegaConf.load("../configs/thicknessNN/base.yaml")
thicknessNN = ApparentThicknessNN(cfg=cfg_thickness)
cfg_bloch = OmegaConf.load("../configs/bloch/base.yaml")
sf_nn = StructureFactorNet(cfg_sf, atoms_nn, thickness_nn=thicknessNN, solve_g_max=cfg_bloch["g_max"])

cfg_bloch = OmegaConf.load("../configs/bloch/base.yaml")
cfg_bloch["use_wave_eq"] = True
bloch_nn = BlochNet(cfg_bloch, sf_nn)


Loaded structure from diffBloch_version_0.0.1/data/non_centro_vs_centro/Ge.cif
Asymmetric unit positions: Parameter containing:
tensor([[0., 0., 0.]], dtype=torch.float64, requires_grad=True)
Atomic numbers: tensor([32], dtype=torch.int32)
Atomic labels: ['Ge1']
ASU Symmetry constraints,(1 if allowed to move): tensor([[0., 0., 0.]], dtype=torch.float64)
Bond constraints: None, None
Angle constraints: None, None
Plane constraints: []
Spacegroup: SpaceGroup #227 (Fd-3m, Cubic). Symmetry matrices: 192, point sym. matr.: 48
Unit cell lattice vectors a,b,c: [[5.65782 0.      0.     ]
 [0.      5.65782 0.     ]
 [0.      0.      5.65782]]
lattice vector lengths: [5.65782 5.65782 5.65782]
Angles (alpha, beta, gamma) between lattice vectors in degrees: [90. 90. 90.]
Thermal displacements: Uij_layer(thermal_displacements=ParameterList(  (0): Parameter containing: [torch.float64 of size ]), constraints=[[(1, 1, 0, 0, 1.0), (2, 2, 0, 0, 1.0)]])
correcting rotation axis position by 0.162 degrees
3

In [44]:
cfg_atoms = OmegaConf.load('../configs/atoms/base.yaml')
cfg_atoms['data']['cif_file_path'] =str(REPO_ROOT / "diffBloch_version_0.0.1/data/non_centro_vs_centro/GaAs.cif") #diffBloch_version_0.0.1/data/quartz/quartz_kinematical_test.cif
atoms_nn = Atoms(cfg_atoms)
#diffBloch_version_0.0.1/data/LuAg/LuAG_IAM.cif

cfg_refinement = OmegaConf.load('../configs/refinement/base.yaml')
cfg_refinement['dataloader']['ignore_orientations'] = []

cfg_refinement['data']['pets_path'] = str(REPO_ROOT / "diffBloch_version_0.0.1/data/non_centro_vs_centro/GaAs.cif_pets") #diffBloch_version_0.0.1/data/quartz/quartz.cif_pets'
cfg_refinement['data']['integration_semiangle'] = 1.5
cfg_refinement['data']['rocking_curve_sampling'] = 60
cfg_refinement['data']['dsg'] = 0.00
cfg_refinement['data']['rsg'] = 0.9
semiangle = cfg_refinement['data']['integration_semiangle'] 

train_dataloader, val_dataloader = get_dataloaders(cfg_refinement, default_thickness=[500])

cfg_sf = OmegaConf.load('../configs/structure_factor/base.yaml')
cfg_thickness = OmegaConf.load('../configs/thicknessNN/base.yaml')
thicknessNN=ApparentThicknessNN(cfg = cfg_thickness)
cfg_bloch = OmegaConf.load('../configs/bloch/base.yaml')
sf_nn = StructureFactorNet(cfg_sf, atoms_nn, thickness_nn=thicknessNN, solve_g_max=cfg_bloch["g_max"])

cfg_bloch = OmegaConf.load('../configs/bloch/base.yaml')
cfg_bloch['use_wave_eq']=True
bloch_nn = BlochNet(cfg_bloch, sf_nn)

Loaded structure from diffBloch_version_0.0.1/data/non_centro_vs_centro/GaAs.cif
Asymmetric unit positions: Parameter containing:
tensor([[0.0000, 0.0000, 0.0000],
        [0.2500, 0.7500, 0.7500]], dtype=torch.float64, requires_grad=True)
Atomic numbers: tensor([31, 33], dtype=torch.int32)
Atomic labels: ['Ga1', 'As1']
ASU Symmetry constraints,(1 if allowed to move): tensor([[0., 0., 0.],
        [0., 0., 0.]], dtype=torch.float64)
Bond constraints: None, None
Angle constraints: None, None
Plane constraints: []
Spacegroup: SpaceGroup #216 (F-43m, Cubic). Symmetry matrices: 96, point sym. matr.: 24
Unit cell lattice vectors a,b,c: [[5.75 0.   0.  ]
 [0.   5.75 0.  ]
 [0.   0.   5.75]]
lattice vector lengths: [5.75 5.75 5.75]
Angles (alpha, beta, gamma) between lattice vectors in degrees: [90. 90. 90.]
Thermal displacements: Uij_layer(thermal_displacements=ParameterList(
    (0): Parameter containing: [torch.float64 of size ]
    (1): Parameter containing: [torch.float64 of size ]
), co

In [48]:
#rot_idx = rotation_idxs[1]
#rot = rotations[1]
tilts = train_dataloader.dataset.rocking_curve_orientations

for batch_idx, batch in enumerate(tqdm(train_dataloader)):
    rotation_idxs, rotations, alphas, thicknesses = batch


import torch

rot = train_dataloader.dataset.rotations[5]

thickness=torch.tensor([500])
results = bloch_nn(rot, tilts = tilts, thickness = thickness)

#results = bloch_nn(rot[:3], tilts = tilts[0:1], thickness = thickness)
# Constants
sg_max = 0.03
g_max = 3.3  # beam cutoff (1/A), as in the paper's Table S1


# Reciprocal cell matrix
reciprocal_cell_matrix = atoms_nn.reciprocal_cell()
initial=0
final= 3

# Rotations and tilts setup
rotations_to_use = train_dataloader.dataset.rotations[initial:final]  # Adjust as necessary

thickness_values = torch.linspace(100, 2500,30)  # Adjust thickness range as needed




100%|██████████| 1/1 [00:00<00:00, 410.36it/s]
diffBloch_version_0.0.1/diffBloch/dynamical.py:515: UserWarning: Theta is None. Expected a valid value for theta.
  warnings.warn("Theta is None. Expected a valid value for theta.")


In [49]:
# Outer loop: Toggle absorption
for absorption in [True, False]:
    # Set the correct folder name based on absorption
    abs_type = "param" if absorption else "no_abs"
    output_dir = Path(f"{project_root}/absorption/non_centro/GaAs")
    os.makedirs(output_dir, exist_ok=True)

    # Update config settings
    cfg_bloch["sg_max"] = float(sg_max)
    cfg_bloch["g_max"] = float(g_max)
    cfg_sf["absorption"] = absorption
    cfg_sf["absorption_type"] = abs_type

    sf_nn = StructureFactorNet(cfg_sf, atoms_nn, thickness_nn=thicknessNN, solve_g_max=cfg_bloch["g_max"])
    bloch_nn = BlochNet(cfg_bloch, sf_nn)

    # Process each rotation
    for idx, rot in enumerate(rotations_to_use, start=initial+1):
        for thickness in tqdm(thickness_values, desc=f"Rotation {idx} ({abs_type})"):
            thickness_value = thickness.item()

            # Run simulation
            results = bloch_nn(rot, tilts=tilts, thickness=[thickness_value])
            results.filter_hkls(200000, cfg_refinement['data']['rsg'], cfg_refinement['data']['dsg'], semiangle)

            # Save rocking curve info (per HKL, all tilts)
            rocking_csv_path = (
                output_dir / f"{abs_type}_rotation_{idx}_thickness_{int(thickness_value)}_rocking_curves.csv"
            )
            intensities, hkls = results.get_integrated_intensities(
                mosaicity=None,  # no rocking-curve smoothing (mosaicity=1 would apply a 5-frame moving average)
            )

            # Convert tensors if necessary
            if isinstance(intensities, torch.Tensor):
                intensities = intensities.detach().cpu().numpy()

            # Apply rotation to the reciprocal matrix and calculate g-vectors
            updated_reciprocal_matrix = reciprocal_cell_matrix @ rot.T
            gvec = hkls @ updated_reciprocal_matrix
            g_vec_length = np.linalg.norm(gvec, axis=1)

            # Collect integrated data
            data = [
                [intensity, " ".join(map(str, hkl)), length]
                for intensity, hkl, length in zip(intensities.flatten(), hkls, g_vec_length)
            ]

            df = pd.DataFrame(data, columns=["Intensity", "HKL", "G-vector Length"])

            # Save integrated intensities
            int_csv_path = (
                output_dir / f"{abs_type}_rotation_{idx}_thickness_{int(thickness_value)}_intensities.csv"
            )
            df.to_csv(int_csv_path, index=False)


diffBloch_version_0.0.1/diffBloch/dynamical.py:340: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(


device:cpu
made it here
U_0_prime not found in StructureFactorNet. Computing it now...
Computed U_0_prime: 0.003393681354452973


Rotation 1 (param):   0%|          | 0/30 [00:00<?, ?it/s]diffBloch_version_0.0.1/diffBloch/diffraction_dataset.py:123: RuntimeWarning: invalid value encountered in divide
  mask1 = d_ewald / sg_max < rsg
Rotation 3 (param): 100%|██████████| 30/30 [00:10<00:00,  2.93it/s]


device:cpu


Rotation 3 (no_abs): 100%|██████████| 30/30 [00:10<00:00,  2.80it/s]
